# Phase 5 — SAM 2.1 GPU validation (Mobile Defect Detection)

Run from top to bottom on **Google Colab T4 GPU**. This notebook does **not** alter your Windows FastAPI app. It uses the original `src/` modules and Phase 4's exported `memory_bank.npz`, `calibration.json`, and `config.json`.

**Requirements**: `phase4-source-only.zip` (or a source ZIP containing `src/`), `msd-model-artifacts.zip`, and a representative phone-screen test image. Do not use an image from the clean reference bank as your only qualitative test.

**Outcome**: baseline-vs-SAM visual comparison, mask areas, region acceptance/fallback stats, and locally downloadable outputs. This is *not* pixel IoU/Dice evaluation without ground-truth masks.

## 1. Verify GPU

In [ ]:
import torch
from pathlib import Path
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Switch Colab to Runtime → Change runtime type → T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))
PROJECT = Path('/content/mobile-defect-detection')
PROJECT.mkdir(parents=True, exist_ok=True)

## 2. Restore or verify project source files
If the required `src/` files are already available in this runtime, skip the upload in the next cell. Otherwise upload your Windows `phase4-source-only.zip`. The extractor normalizes `\` paths to Linux separators and ignores compiled files. Do not upload the model artifact ZIP here.

In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import zipfile

source_ok = (PROJECT/'src/inference.py').is_file() and (PROJECT/'src/data/msd_dataset.py').is_file()
if not source_ok:
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload one source ZIP.'
    filename = next(iter(uploaded))
    with zipfile.ZipFile(filename) as z:
        count = 0
        for info in z.infolist():
            parts = PurePosixPath(info.filename.replace('\\','/')).parts
            if 'src' not in parts or info.is_dir(): continue
            rel = parts[parts.index('src'):]
            if '__pycache__' in rel or '__MACOSX' in rel or rel[-1].endswith(('.pyc','.pyo')): continue
            if any(part in ('..','') for part in rel): continue
            dest = PROJECT.joinpath(*rel)
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as inp, dest.open('wb') as out:
                import shutil
                shutil.copyfileobj(inp,out)
            count += 1
    print('Extracted source files:',count)
else:
    print('Existing source files found. Upload skipped.')
assert (PROJECT/'src/inference.py').is_file()
assert (PROJECT/'src/data/msd_dataset.py').is_file()
print('Source ready:',PROJECT)

## 3. Install dependencies
The first run downloads pretrained weights. The official Meta SAM 2 package is installed from its repository; if Colab asks for a restart, restart the runtime, then re-run relevant setup cells.

In [ ]:
%pip -q install 'numpy>=1.24' 'Pillow>=10' 'scikit-learn>=1.3' 'scikit-image>=0.22' 'scipy>=1.11' 'opencv-python>=4.8' huggingface_hub
%pip -q install 'git+https://github.com/facebookresearch/sam2.git' 

## 4. Restore your Phase 4 artifacts
Upload `msd-model-artifacts.zip` from Phase 4. That ZIP should contain the three files at its root (possibly with a parent folder).

In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import zipfile
from pathlib import Path
import shutil

OUT_ART = PROJECT/'artifacts/msd'
OUT_ART.mkdir(parents=True,exist_ok=True)
expected = ('memory_bank.npz','calibration.json','config.json')
if not all((OUT_ART/x).is_file() for x in expected):
    uploaded = files.upload()
    assert len(uploaded)==1, 'Upload the artifacts ZIP, one file only.'
    name=next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        for needed in expected:
            match=[v for v in z.infolist() if v.filename.replace('\\','/').split('/')[-1]==needed and not v.is_dir()]
            if len(match)!=1:
                raise FileNotFoundError(f'Expected one {needed}; got {len(match)}')
            with z.open(match[0]) as inp, (OUT_ART/needed).open('wb') as output:
                shutil.copyfileobj(inp,output)
for item in expected:
    p=OUT_ART/item
    assert p.is_file() and p.stat().st_size>0, f'Missing/empty: {p}'
    print(item,p.stat().st_size,'bytes')

## 5. Download SAM 2.1 small checkpoint
The checkpoint and config must match. We use Meta’s `facebook/sam2.1-hiera-small` repository and the small (`hiera_s`) configuration.

In [ ]:
from huggingface_hub import hf_hub_download
from sam2.build_sam import build_sam2

checkpoint = hf_hub_download(repo_id='facebook/sam2.1-hiera-small', filename='sam2.1_hiera_small.pt')
model_config = 'configs/sam2.1/sam2.1_hiera_s.yaml'
print('Checkpoint:',checkpoint)
print('Config:',model_config)
print('Checkpoint exists:',Path(checkpoint).is_file())

## 6. Load your original DINOv2+memory bank and SAM 2 wrapper

In [ ]:
import sys, json
sys.path.insert(0,str(PROJECT))
from src.config import PipelineConfig
from src.calibration.calibration import Calibration
from src.memory_bank.memory_bank import MemoryBank
from src.feature_extraction.dinov2_extractor import DinoV2Extractor
from src.inference import DefectInspector
from src.sam2_prompt.sam2_predictor import Sam2DefectSegmenter
cfg=PipelineConfig.from_dict(json.loads((OUT_ART/'config.json').read_text()))
cal=Calibration.from_dict(json.loads((OUT_ART/'calibration.json').read_text()))
extractor=DinoV2Extractor.from_config(cfg)
bank=MemoryBank.load(str(OUT_ART/'memory_bank.npz'))
segmenter=Sam2DefectSegmenter(checkpoint_path=checkpoint,model_cfg=model_config,device='cuda')
inspector=DefectInspector(extractor,bank,cal,cfg,segmenter=segmenter,roi_method=None)
print('Model and SAM 2 initialized.')

## 7. Upload a representative smartphone image

In [ ]:
from PIL import Image,ImageOps
import io
uploaded=files.upload()
assert len(uploaded)==1,'Select one image (JPG/PNG/WebP).'
name, image_bytes=next(iter(uploaded.items()))
image=ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes))).convert('RGB')
print('Test image:',name,'Dimensions:',image.size)

## 8. Run baseline and SAM 2.1 on the *same* input
`run_sam=False` produces the calibrated heatmap mask. `run_sam=True` attempts SAM refinement when candidate regions exist. For normal images or no candidates, `sam_mask` can legitimately be `None`.

In [ ]:
import time
with torch.inference_mode():
    t0=time.perf_counter()
    baseline=inspector.inspect(image,run_sam=False)
    baseline_secs=time.perf_counter()-t0
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.perf_counter()
    sam_result=inspector.inspect(image,run_sam=True)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    sam_secs=time.perf_counter()-t0
print('Prediction:', 'DEFECTIVE' if baseline.is_defective else 'NORMAL')
print('Image score:',round(baseline.image_score,4),'Threshold:',round(cal.image_threshold,4))
print('Candidate regions:',len(baseline.regions))
print('SAM mask produced:',sam_result.sam_mask is not None)
print('Baseline mask coverage:',round(100*float((baseline.final_mask>0).mean()),3),'%')
print('Final mask coverage:',round(100*float((sam_result.final_mask>0).mean()),3),'%')
print('Run durations (seconds):',round(baseline_secs,2),round(sam_secs,2))
if sam_result.segmentation is not None:
    print('SAM region attempts:',len(sam_result.segmentation.per_region))
    print('Accepted:',sum(r.accepted for r in sam_result.segmentation.per_region))
    print('Fallbacks:',sam_result.segmentation.n_fallbacks)
else:
    print('No SAM region segmentation; normal/no-region cases do not use SAM.')

## 9. Display heatmap and masks
These images are qualitative, not numerical evaluation. Do not report IoU or Dice without ground-truth labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

fig,axs=plt.subplots(1,4,figsize=(20,5))
axs[0].imshow(sam_result.image);axs[0].set_title('Image')
axs[1].imshow(sam_result.heatmap,cmap='turbo');axs[1].set_title('Raw anomaly heatmap (display colormap)')
axs[2].imshow(baseline.final_mask,cmap='gray');axs[2].set_title('Baseline heatmap mask')
axs[3].imshow(sam_result.final_mask,cmap='gray');axs[3].set_title('SAM attempt / region fallback')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.show()

## 10. Save outputs and download a report ZIP

In [ ]:
import json, shutil
from PIL import Image
import numpy as np
save_dir=PROJECT/'results/phase5'
save_dir.mkdir(parents=True,exist_ok=True)
sam_result.image.save(save_dir/'input_working.png')
Image.fromarray(((baseline.final_mask>0).astype(np.uint8)*255)).save(save_dir/'baseline_mask.png')
Image.fromarray(((sam_result.final_mask>0).astype(np.uint8)*255)).save(save_dir/'sam_final_mask.png')
np.save(save_dir/'raw_heatmap.npy',np.asarray(sam_result.heatmap,dtype=np.float32))
per_region=sam_result.segmentation.per_region if sam_result.segmentation is not None else []
report={
  'prediction':'defective' if sam_result.is_defective else 'normal',
  'image_score':float(sam_result.image_score),
  'image_threshold':float(cal.image_threshold),
  'pixel_threshold':float(cal.pixel_threshold),
  'regions':len(sam_result.regions),
  'sam_mask_produced':sam_result.sam_mask is not None,
  'sam_region_attempts':len(per_region),
  'sam_accepted_regions':sum(r.accepted for r in per_region),
  'sam_fallback_regions':sum(r.used_fallback for r in per_region),
  'baseline_area_percent':100*float((baseline.final_mask>0).mean()),
  'final_area_percent':100*float((sam_result.final_mask>0).mean()),
  'baseline_seconds':baseline_secs,
  'sam_seconds':sam_secs,
  'model_config':model_config,
  'warning':sam_result.info.get('warning')
}
(save_dir/'phase5_report.json').write_text(json.dumps(report,indent=2))
archive_path=shutil.make_archive('/content/phase5-sam-comparison','zip',root_dir=save_dir)
print('Created:',archive_path)
files.download(archive_path)

## Next: integrate SAM 2 into FastAPI

First review whether SAM improves the segmentation visually **and** records accepted vs fallback regions. Your existing Phase 4 `model_service.py` supports `ENABLE_SAM2=1`, `SAM2_CHECKPOINT`, and `SAM2_CONFIG`. However, **Colab GPU and Windows FastAPI are separate environments**: this notebook alone does not enable SAM in the website. Before doing so, decide whether to run SAM locally with suitable hardware or host GPU inference separately. Do not commit checkpoint files or uploaded images to Git.